# Final Project NLP - Fact Checking Pipeline

Pipeline: **Task 1** claim detection → **Task 2** evidence retrieval + rerank → **Task 3** verifikasi pakai NLI
(SUPPORTS / REFUTES / NOT ENOUGH INFO).

Catatan:
- Runtime pakai GPU (T4).
- Semua yang perlu disimpan (model, data, hasil eval) ada di Google Drive, folder `DRIVE_DIR` di bawah.
- Model task 1 dicari di folder `models/` di Drive. Kalau belum ada, ditrain di sini (~15 menit).
- Run pertama agak lama (~30 menit) karena download `wiki-pages.zip` 1.7 GB terus bikin subset corpus. Run berikutnya langsung skip.

## Setup

In [ ]:
import os
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/NLP/Praktikum/Final Project')

RAW_DIR = DRIVE_DIR / 'data' / 'raw'
PROCESSED_DIR = DRIVE_DIR / 'data' / 'processed'
MODELS_DIR = DRIVE_DIR / 'models'
OUTPUTS_DIR = DRIVE_DIR / 'outputs'

# cache model huggingface di drive biar ga download ulang tiap session
os.environ['HF_HOME'] = str(DRIVE_DIR / 'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

for d in [RAW_DIR, PROCESSED_DIR, MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
!ls "{DRIVE_DIR}"

In [ ]:
!nvidia-smi -L
!pip install -q "sentence-transformers>=3.0" accelerate sentencepiece protobuf
!python -m spacy download en_core_web_sm 2>&1 | tail -1

In [ ]:
import json
import random
import re
import time
import unicodedata
import urllib.request
import zipfile
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device, torch.__version__, transformers.__version__)

## Data
Fungsi buat download dataset. File ditulis ke `.part` dulu, baru di-rename kalau downloadnya selesai,
jadi kalau putus di tengah ga dianggap udah ke-download.

In [ ]:
FEVER_URL = 'https://fever.ai/download/fever'
CLAIMBUSTER_URL = 'https://zenodo.org/records/3609356/files'
CLAIMBUSTER_DIR = RAW_DIR / 'claimbuster'


def download(url, dest):
    dest = Path(dest)
    if dest.exists():
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + '.part')

    req = urllib.request.Request(url, headers={'User-Agent': 'fact-checking-pipeline/0.1'})
    with urllib.request.urlopen(req, timeout=60) as r, open(tmp, 'wb') as f:
        size = int(r.headers.get('Content-Length', 0)) or None
        with tqdm(total=size, unit='B', unit_scale=True, desc=dest.name) as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    tmp.rename(dest)
    return dest


def download_fever_file(name):
    return download(f'{FEVER_URL}/{name}', RAW_DIR / name)


def download_claimbuster_file(name):
    return download(f'{CLAIMBUSTER_URL}/{name}?download=1', CLAIMBUSTER_DIR / name)

## Task 1 - Claim Detection
Teks dipecah per kalimat pakai spaCy, terus tiap kalimat diklasifikasi claim / bukan claim pakai RoBERTa
yang udah difinetune di ClaimBuster. Kalimat yang lolos threshold doang yang lanjut ke task 2.
Threshold-nya hasil tuning di data val, disimpan di `threshold.json`.

In [ ]:
class ClaimDetector:
    def __init__(self, model_dir, threshold=None, batch_size=64, max_length=128):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        import spacy

        model_dir = Path(model_dir)
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
        self.nlp = spacy.load('en_core_web_sm', exclude=['ner', 'lemmatizer', 'attribute_ruler', 'tagger'])
        self.batch_size = batch_size
        self.max_length = max_length

        if threshold is None:
            thr_file = model_dir / 'threshold.json'
            threshold = json.loads(thr_file.read_text())['threshold'] if thr_file.exists() else 0.5
        self.threshold = float(threshold)

    def split_sentences(self, text):
        return [s.text.strip() for s in self.nlp(text).sents if s.text.strip()]

    @torch.no_grad()
    def score(self, sentences):
        probs = []
        for i in range(0, len(sentences), self.batch_size):
            batch = self.tokenizer(sentences[i:i + self.batch_size], truncation=True, max_length=self.max_length,
                                   padding=True, return_tensors='pt').to(device)
            logits = self.model(**batch).logits.float()
            probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
        return np.concatenate(probs) if probs else np.zeros(0)

    def detect(self, text, return_all=False):
        sentences = self.split_sentences(text)
        results = []
        for i, (s, p) in enumerate(zip(sentences, self.score(sentences))):
            results.append({'index': i, 'sentence': s, 'p_claim': float(p), 'is_claim': bool(p >= self.threshold)})
        if return_all:
            return results
        return [r for r in results if r['is_claim']]

### Training task 1
Cuma kepake kalau model belum ada di Drive.

Data ClaimBuster: `crowdsourced.csv` buat train/val, `groundtruth.csv` buat test.
Label utama (`label`): CFS + UFS = 1 (claim), NFS = 0. Ada juga `label_checkworthy` (CFS doang = 1) buat perbandingan.
Kalimat yang ada juga di test dibuang dari train/val, dan kalimat duplikat yang labelnya beda dibuang juga.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

LABEL_COLS = ['label', 'label_checkworthy']


def load_claimbuster(path):
    df = pd.read_csv(path)
    df = df.rename(columns={'Sentence_id': 'sentence_id', 'Text': 'text', 'Verdict': 'verdict',
                            'Speaker': 'speaker', 'File_id': 'file_id'})
    df['text'] = df['text'].astype(str).str.strip()
    df = df[df['text'] != ''].copy()
    df['verdict'] = df['verdict'].astype(int)
    df['label'] = (df['verdict'] >= 0).astype(int)
    df['label_checkworthy'] = (df['verdict'] == 1).astype(int)
    df['norm'] = df['text'].str.lower().str.split().str.join(' ')
    return df


def build_task1_data(val_size=0.1, seed=42):
    crowd = load_claimbuster(CLAIMBUSTER_DIR / 'crowdsourced.csv')
    test = load_claimbuster(CLAIMBUSTER_DIR / 'groundtruth.csv')

    # buang kalimat yang juga ada di test set
    crowd = crowd[~crowd['norm'].isin(set(test['norm']))]
    # duplikat: kalau labelnya beda-beda buang semua, kalau sama ambil satu
    n_verdicts = crowd.groupby('norm')['verdict'].nunique()
    conflict = set(n_verdicts[n_verdicts > 1].index)
    crowd = crowd[~crowd['norm'].isin(conflict)].drop_duplicates('norm')

    train, val = train_test_split(crowd, test_size=val_size, random_state=seed, stratify=crowd['verdict'])

    # class weight = n / (n_class * count), biar loss-nya seimbang
    weights = {}
    for col in LABEL_COLS:
        counts = train[col].value_counts().sort_index()
        weights[col] = [len(train) / (2 * counts[c]) for c in counts.index]

    cols = ['sentence_id', 'text', 'verdict', 'label', 'label_checkworthy', 'speaker', 'file_id']
    for name, df in [('train', train), ('val', val), ('test', test)]:
        df[cols].reset_index(drop=True).to_parquet(PROCESSED_DIR / f'task1_{name}.parquet', index=False)
        print(name, len(df), 'positive rate:', round(df['label'].mean(), 3))
    (PROCESSED_DIR / 'task1_meta.json').write_text(json.dumps({'seed': seed, 'class_weights': weights}, indent=2))


def load_task1_split(name, label_col):
    df = pd.read_parquet(PROCESSED_DIR / f'task1_{name}.parquet')
    return df, df[label_col].to_numpy()


class SentenceDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.enc = tokenizer(list(texts), truncation=True, max_length=max_length)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item['labels'] = int(self.labels[i])
        return item


class WeightedTrainer(transformers.Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits, labels, weight=self.class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss


class SaveBestModel(transformers.TrainerCallback):
    # simpan model tiap kali macro F1 val naik, jadi di disk cuma ada 1 copy (yang terbaik)
    def __init__(self, save_dir):
        self.save_dir = save_dir
        self.best_score = -1
        self.best_step = None

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        score = metrics.get('eval_macro_f1')
        if score is not None and score > self.best_score:
            self.best_score = score
            self.best_step = state.global_step
            model.save_pretrained(self.save_dir)


def p_claim(logits):
    return torch.softmax(torch.as_tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()


def binary_metrics(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
        'accuracy': accuracy_score(labels, preds),
        'roc_auc': roc_auc_score(labels, probs) if len(set(labels)) > 1 else float('nan'),
        'threshold': float(threshold),
    }


def tune_threshold(labels, probs):
    # cari threshold yang macro F1-nya paling tinggi
    grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
    scores = [f1_score(labels, (probs >= t).astype(int), average='macro') for t in grid]
    return float(grid[np.argmax(scores)])


def train_task1(model_name='roberta-base', label_col='label', epochs=3, lr=2e-5, batch_size=32,
                eval_steps=200, seed=42, max_steps=-1):
    args_used = dict(locals())
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, TrainingArguments

    transformers.set_seed(seed)
    run_name = f"task1_{model_name.split('/')[-1]}_{label_col}"
    out_dir = MODELS_DIR / run_name
    model_dir = out_dir / 'model'
    out_dir.mkdir(parents=True, exist_ok=True)

    class_weights = json.loads((PROCESSED_DIR / 'task1_meta.json').read_text())['class_weights'][label_col]
    print(run_name, 'class weights:', class_weights)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, id2label={0: 'non_factual', 1: 'claim'}, label2id={'non_factual': 0, 'claim': 1})
    model = model.float()  # beberapa checkpoint kesimpen fp16, harus fp32 biar fp16 training ga error

    splits = {name: load_task1_split(name, label_col) for name in ['train', 'val', 'test']}
    data = {name: SentenceDataset(df['text'], y, tokenizer) for name, (df, y) in splits.items()}

    total_steps = max_steps if max_steps > 0 else int(np.ceil(len(data['train']) / batch_size) * epochs)
    eval_steps = min(eval_steps, max(1, total_steps // 2))

    training_args = TrainingArguments(
        output_dir=f'/content/trainer_tmp/{run_name}',   # file sementara trainer, ga usah ke drive
        num_train_epochs=epochs,
        max_steps=max_steps,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        warmup_steps=int(0.06 * total_steps),
        weight_decay=0.01,
        eval_strategy='steps',
        eval_steps=eval_steps,
        logging_steps=max(1, eval_steps // 4),
        save_strategy='no',
        report_to='none',
        seed=seed,
        fp16=torch.cuda.is_available(),
    )
    keep_best = SaveBestModel(model_dir)
    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=data['train'],
        eval_dataset=data['val'],
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=lambda ev: binary_metrics(np.asarray(ev[1]), p_claim(ev[0])),
        callbacks=[keep_best],
        class_weights=class_weights,
    )
    trainer.train()
    print(f'best val macro F1 {keep_best.best_score:.4f} (step {keep_best.best_step})')

    # load lagi model terbaik, terus tuning threshold di val
    best = AutoModelForSequenceClassification.from_pretrained(model_dir)
    trainer.model.load_state_dict(best.state_dict())
    val_y, test_y = splits['val'][1], splits['test'][1]
    val_probs = p_claim(trainer.predict(data['val']).predictions)
    test_probs = p_claim(trainer.predict(data['test']).predictions)
    threshold = tune_threshold(val_y, val_probs)

    results = {
        'run': run_name,
        'args': args_used,
        'class_weights': class_weights,
        'best_step': keep_best.best_step,
        'tuned_threshold': threshold,
        'val@0.5': binary_metrics(val_y, val_probs, 0.5),
        'val@tuned': binary_metrics(val_y, val_probs, threshold),
        'test@0.5': binary_metrics(test_y, test_probs, 0.5),
        'test@tuned': binary_metrics(test_y, test_probs, threshold),
        'log_history': trainer.state.log_history,
    }
    tokenizer.save_pretrained(model_dir)
    (model_dir / 'threshold.json').write_text(json.dumps({'threshold': threshold, 'label_col': label_col}))
    (out_dir / 'metrics.json').write_text(json.dumps(results, indent=2, default=float))

    test_df = splits['test'][0][['sentence_id', 'text', 'verdict', label_col]].copy()
    test_df['p_claim'] = test_probs
    test_df['pred'] = (test_probs >= threshold).astype(int)
    test_df.to_csv(out_dir / 'test_predictions.csv', index=False)

    print('threshold:', threshold)
    for k in ['val@0.5', 'val@tuned', 'test@0.5', 'test@tuned']:
        m = results[k]
        print(f"{k:<11} macroF1={m['macro_f1']:.4f}  P={m['precision']:.4f}  R={m['recall']:.4f}  AUC={m['roc_auc']:.4f}")
    return results

### Load model task 1
Cari model di semua subfolder `models/` (folder yang ada `model.safetensors` + `config.json`).
Kalau ada lebih dari satu, ambil yang ada `threshold.json` dan `metrics.json`-nya.
Kalau ga ketemu sama sekali, train dulu.

In [ ]:
TRAIN_IF_MISSING = True  # @param {type:"boolean"}


def is_valid_safetensors(path):
    # 8 byte pertama = panjang header JSON. kalau file kepotong / upload ke drive belum kelar,
    # angkanya ngaco dan pas di-load keluar "header too large"
    try:
        size = path.stat().st_size
        with open(path, 'rb') as f:
            n = int.from_bytes(f.read(8), 'little')
            if n > size - 8:
                return False
            json.loads(f.read(n))
        return True
    except (OSError, ValueError):
        return False


def find_task1_models():
    found = []
    for p in MODELS_DIR.rglob('model.safetensors'):
        if not (p.parent / 'config.json').exists():
            continue
        if not is_valid_safetensors(p):
            print(f'file rusak, skip: {p} ({p.stat().st_size / 1e6:.1f} MB)')
            continue
        found.append(p.parent)
    # urutan: ada threshold.json, ada metrics.json, paling baru
    found.sort(key=lambda d: ((d / 'threshold.json').exists(), (d.parent / 'metrics.json').exists(),
                              (d / 'model.safetensors').stat().st_mtime), reverse=True)
    return found


models_found = find_task1_models()

# zip dari notebook task1 yang lama
zip_path = DRIVE_DIR / 'task1_roberta-base_label.zip'
if not models_found and zip_path.exists():
    print('extract', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DRIVE_DIR)
    models_found = find_task1_models()

if not models_found:
    assert TRAIN_IF_MISSING, f'model task 1 ga ketemu di {MODELS_DIR}'
    print('model belum ada, train dulu...')
    for f in ['groundtruth.csv', 'crowdsourced.csv']:
        download_claimbuster_file(f)
    build_task1_data()
    train_task1('roberta-base', 'label')
    models_found = find_task1_models()

for d in models_found:
    print(d, f"({(d / 'model.safetensors').stat().st_size / 1e6:.1f} MB)")
TASK1_MODEL_DIR = models_found[0]
print('pakai:', TASK1_MODEL_DIR)

In [ ]:
detector = ClaimDetector(TASK1_MODEL_DIR)
print('threshold:', detector.threshold)

print('\n===== TASK 1: claim detection =====')
for r in detector.detect('Hello everyone! I think pizza is great. Barack Obama was born in Hawaii in 1961.', return_all=True):
    print('CLAIM' if r['is_claim'] else '  -  ', f"{r['p_claim']:.3f}", r['sentence'])

## Task 2 - Evidence Retrieval & Reranking
Alurnya:
1. BM25 (judul + isi halaman) → top 5 halaman
2. bi-encoder `all-MiniLM-L6-v2` → top 30 kalimat
3. cross-encoder `ms-marco-MiniLM-L-6-v2` → rerank, ambil top 5

In [ ]:
PTB_TOKENS = {'-LRB-': '(', '-RRB-': ')', '-LSB-': '[', '-RSB-': ']', '-LCB-': '{', '-RCB-': '}',
              '-COLON-': ':', '``': '"', "''": '"'}


def normalize_page_id(page_id):
    return unicodedata.normalize('NFC', page_id)


def clean_text(text):
    # FEVER pakai token PTB kayak -LRB- buat kurung
    for tok, ch in PTB_TOKENS.items():
        text = text.replace(tok, ch)
    text = text.replace('( ', '(').replace(' )', ')')
    return ' '.join(text.split())


def title_from_page_id(page_id):
    return clean_text(page_id.replace('_', ' '))


def load_claims(path, n=None, seed=42, include_nei=False):
    claims = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            if row['label'] == 'NOT ENOUGH INFO' and not include_nei:
                continue
            evidence_sets = []
            for ev_set in row['evidence']:
                sents = tuple(sorted({(normalize_page_id(page), int(line_id))
                                      for _, _, page, line_id in ev_set if page is not None}))
                if sents and sents not in evidence_sets:
                    evidence_sets.append(sents)
            claims.append({'id': row['id'], 'claim': row['claim'], 'label': row['label'],
                           'evidence_sets': [list(s) for s in evidence_sets]})
    if n is not None and n < len(claims):
        claims = random.Random(seed).sample(claims, n)
    return claims


def save_jsonl(rows, path):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def read_claims_cache(path):
    claims = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            c = json.loads(line)
            c['evidence_sets'] = [[tuple(s) for s in ev] for ev in c['evidence_sets']]
            claims.append(c)
    return claims


def iter_wiki_pages(zip_path):
    # baca langsung dari zip, ga usah di-extract
    with zipfile.ZipFile(zip_path) as zf:
        for name in sorted(zf.namelist()):
            if not name.endswith('.jsonl') or name.startswith('__MACOSX'):
                continue
            with zf.open(name) as f:
                for line in f:
                    page = json.loads(line)
                    if page['id']:
                        yield page


def parse_lines(lines):
    sentences = []
    for row in lines.split('\n'):
        parts = row.split('\t')
        if len(parts) < 2 or not parts[1].strip() or not parts[0].isdigit():
            continue
        sentences.append((int(parts[0]), parts[1]))
    return sentences

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer


class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.vectorizer = CountVectorizer(lowercase=True, stop_words='english',
                                          token_pattern=r'(?u)\b\w+\b', dtype=np.float32)

    def fit(self, texts):
        tf = self.vectorizer.fit_transform(texts).tocsr()
        n_docs = tf.shape[0]
        doc_len = np.asarray(tf.sum(axis=1)).ravel()
        avgdl = doc_len.mean()

        df = np.bincount(tf.indices, minlength=tf.shape[1])
        self.idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)

        # bobot tf BM25 dihitung sekali di sini, jadi pas query tinggal dikali idf
        len_per_value = np.repeat(doc_len, np.diff(tf.indptr))
        tf.data = tf.data * (self.k1 + 1) / (tf.data + self.k1 * (1 - self.b + self.b * len_per_value / avgdl))
        self.matrix = tf.tocsc()
        return self

    def get_scores(self, query):
        q = self.vectorizer.transform([query])
        if len(q.indices) == 0:
            return np.zeros(self.matrix.shape[0], dtype=np.float32)
        weights = self.idf[q.indices] * q.data
        return np.asarray(self.matrix[:, q.indices] @ weights).ravel()


def top_k_indices(scores, k):
    k = min(k, len(scores))
    if k == 0:
        return []
    idx = np.argpartition(-scores, k - 1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return [(int(i), float(scores[i])) for i in idx]

In [ ]:
@dataclass
class Evidence:
    page_id: str
    line_id: int
    title: str
    text: str
    doc_score: float = 0.0
    retrieval_score: float = 0.0
    rerank_score: float = None

    @property
    def key(self):
        return (self.page_id, self.line_id)

    def to_dict(self):
        return asdict(self)


def load_corpus(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]


class EvidenceRetriever:
    def __init__(self, pages, bi_encoder='sentence-transformers/all-MiniLM-L6-v2', title_weight=2.0):
        self.pages = pages
        self.title_weight = title_weight
        print(f'bikin index BM25 ({len(pages)} halaman)...')
        self.title_index = BM25().fit([p['title'] for p in pages])
        self.body_index = BM25().fit([' '.join(s[1] for s in p['sentences']) for p in pages])
        self.bi_encoder = bi_encoder
        self._encoder = None

    @property
    def encoder(self):
        if self._encoder is None:
            from sentence_transformers import SentenceTransformer
            self._encoder = SentenceTransformer(self.bi_encoder, device=device)
        return self._encoder

    # tahap 1: cari dokumen
    def retrieve_documents(self, claim, k=5):
        scores = self.title_weight * self.title_index.get_scores(claim) + self.body_index.get_scores(claim)
        return top_k_indices(scores, k)

    # tahap 2: ambil semua kalimat dari top-k dokumen, urutkan pakai bi-encoder
    def retrieve(self, claim, top_k=30, doc_k=5):
        candidates = []
        for idx, doc_score in self.retrieve_documents(claim, doc_k):
            if doc_score <= 0:
                continue
            page = self.pages[idx]
            for line_id, text in page['sentences']:
                candidates.append(Evidence(page['id'], int(line_id), page['title'], text, doc_score=doc_score))
        if not candidates:
            return []

        claim_emb = self.encoder.encode([claim], normalize_embeddings=True)
        sent_emb = self.encoder.encode([f'{c.title}: {c.text}' for c in candidates],
                                       normalize_embeddings=True, batch_size=64)
        for c, sim in zip(candidates, sent_emb @ claim_emb[0]):
            c.retrieval_score = float(sim)
        candidates.sort(key=lambda c: c.retrieval_score, reverse=True)
        return candidates[:top_k]


# tahap 3: rerank pakai cross-encoder
class CrossEncoderReranker:
    def __init__(self, model_name='cross-encoder/ms-marco-MiniLM-L-6-v2', batch_size=32):
        from sentence_transformers import CrossEncoder
        self.model = CrossEncoder(model_name, device=device)
        self.batch_size = batch_size

    def rerank(self, claim, candidates, top_k=5):
        if not candidates:
            return []
        pairs = [(claim, f'{c.title}: {c.text}') for c in candidates]
        scores = self.model.predict(pairs, batch_size=self.batch_size, show_progress_bar=False)
        for c, s in zip(candidates, scores):
            c.rerank_score = float(s)
        return sorted(candidates, key=lambda c: c.rerank_score, reverse=True)[:top_k]


class EvidencePipeline:
    def __init__(self, retriever, reranker=None, doc_k=5, n_candidates=30):
        self.retriever = retriever
        self.reranker = reranker
        self.doc_k = doc_k
        self.n_candidates = n_candidates

    def run(self, claim, top_k=5):
        candidates = self.retriever.retrieve(claim, top_k=self.n_candidates, doc_k=self.doc_k)
        if self.reranker:
            results = self.reranker.rerank(claim, candidates, top_k=top_k)
        else:
            results = candidates[:top_k]
        return [r.to_dict() for r in results]

### Bikin subset corpus Wikipedia
Index 5.4 juta halaman kegedean, jadi corpus-nya subset:
- **gold**: halaman evidence dari 500 claim sampel
- **hard negative**: halaman yang judulnya muncul di claim (misal "Soul Food" ikut narik "Soul Food (TV series)")
- **random**: halaman acak buat noise

Catatan: recall di subset ini lebih optimis dibanding full Wikipedia, dan claim di luar sampel biasanya ga ketemu evidence-nya.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

CORPUS_PATH = PROCESSED_DIR / 'wiki_subset.jsonl'
CLAIMS_SAMPLE_PATH = PROCESSED_DIR / 'claims_sample.jsonl'
TOTAL_WIKI_PAGES = 5_416_537


def claim_ngrams(claims, max_n=3):
    grams = set()
    for c in claims:
        tokens = re.findall(r'\w+', c['claim'].lower())
        for n in range(1, max_n + 1):
            for i in range(len(tokens) - n + 1):
                gram = tokens[i:i + n]
                if n == 1 and (gram[0] in ENGLISH_STOP_WORDS or len(gram[0]) < 3):
                    continue
                grams.add(' '.join(gram))
    return grams


def base_title(title):
    # "Soul Food (TV series)" -> "soul food"
    title = re.sub(r'\s*\(.*\)\s*$', '', title)
    return ' '.join(re.findall(r'\w+', title.lower()))


def build_corpus(n_claims=500, n_random=20000, seed=42):
    claims = load_claims(download_fever_file('shared_task_dev.jsonl'), n=n_claims, seed=seed)
    zip_path = download_fever_file('wiki-pages.zip')
    print('jumlah claim:', len(claims))

    gold_pages = {page for c in claims for ev in c['evidence_sets'] for page, _ in ev}
    grams = claim_ngrams(claims)
    rng = random.Random(seed)

    corpus = []
    counts = {'gold': 0, 'hard_negative': 0, 'random': 0}
    for page in tqdm(iter_wiki_pages(zip_path), total=TOTAL_WIKI_PAGES):
        page_id = normalize_page_id(page['id'])
        title = title_from_page_id(page_id)
        if page_id in gold_pages:
            source = 'gold'
        elif base_title(title) in grams:
            source = 'hard_negative'
        elif rng.random() < n_random / TOTAL_WIKI_PAGES:
            source = 'random'
        else:
            continue
        sentences = parse_lines(page['lines'])
        if sentences:
            corpus.append({'id': page_id, 'title': title, 'source': source,
                           'sentences': [[i, clean_text(t)] for i, t in sentences]})
            counts[source] += 1

    print('jumlah halaman:', len(corpus), counts)
    missing = gold_pages - {p['id'] for p in corpus if p['source'] == 'gold'}
    if missing:
        print(len(missing), 'gold page ga ketemu, contoh:', list(missing)[:5])
    save_jsonl(claims, CLAIMS_SAMPLE_PATH)
    save_jsonl(corpus, CORPUS_PATH)


if CORPUS_PATH.exists() and CLAIMS_SAMPLE_PATH.exists():
    print('corpus udah ada, skip')
else:
    build_corpus()
!ls -lh "{PROCESSED_DIR}"

In [ ]:
evidence_pipe = EvidencePipeline(EvidenceRetriever(load_corpus(CORPUS_PATH)), CrossEncoderReranker())
sample_claims = read_claims_cache(CLAIMS_SAMPLE_PATH)

# contoh pakai claim dari sampel, karena corpus-nya cuma nyakup claim-claim itu
demo = sample_claims[0]
print('\n===== TASK 2: evidence retrieval =====')
print('claim:', demo['claim'], f"({demo['label']})")
demo_evidence = evidence_pipe.run(demo['claim'])
for i, ev in enumerate(demo_evidence, 1):
    print(f"{i}. [{ev['title']} #{ev['line_id']}] {ev['rerank_score']:.2f}  {ev['text']}")

### Evaluasi task 2
- doc recall@k: semua halaman dari salah satu evidence set masuk top-k BM25
- candidate recall: evidence set lengkap ada di 30 kandidat (batas atas buat reranker)
- evidence recall@5 (metrik resmi FEVER), hit@1, MRR, dibandingin sebelum vs sesudah rerank

In [ ]:
def full_set_found(evidence_sets, found_keys):
    found_keys = set(found_keys)
    return any(all(tuple(s) in found_keys for s in ev) for ev in evidence_sets)


def reciprocal_rank(ranked, gold):
    for rank, key in enumerate(ranked, 1):
        if key in gold:
            return 1 / rank
    return 0.0


def evaluate_retrieval(claims, retriever, reranker=None, doc_k=5, n_candidates=30, top_k=5):
    totals = {}
    predictions = []

    def add(name, value):
        totals[name] = totals.get(name, 0) + float(value)

    for c in tqdm(claims):
        gold_sets = c['evidence_sets']
        gold_keys = {tuple(s) for ev in gold_sets for s in ev}

        docs = [retriever.pages[i]['id'] for i, _ in retriever.retrieve_documents(c['claim'], k=10)]
        for k in [1, 5, 10]:
            add(f'doc_recall@{k}', any(all(p in docs[:k] for p, _ in ev) for ev in gold_sets))

        candidates = retriever.retrieve(c['claim'], top_k=n_candidates, doc_k=doc_k)
        before = [x.key for x in candidates]
        add('candidate_recall', full_set_found(gold_sets, before))
        add('before/evidence_recall@5', full_set_found(gold_sets, before[:top_k]))
        add('before/hit@1', bool(before) and before[0] in gold_keys)
        add('before/mrr', reciprocal_rank(before, gold_keys))

        final = candidates[:top_k]
        if reranker is not None:
            reranked = reranker.rerank(c['claim'], candidates, top_k=len(candidates))
            after = [x.key for x in reranked]
            add('after/evidence_recall@5', full_set_found(gold_sets, after[:top_k]))
            add('after/hit@1', bool(after) and after[0] in gold_keys)
            add('after/mrr', reciprocal_rank(after, gold_keys))
            final = reranked[:top_k]

        predictions.append({'id': c['id'], 'claim': c['claim'], 'label': c['label'], 'gold_evidence': gold_sets,
                            'predicted_evidence': [x.to_dict() for x in final]})

    return {k: v / len(claims) for k, v in totals.items()}, predictions


def print_retrieval_metrics(m):
    for k in ['doc_recall@1', 'doc_recall@5', 'doc_recall@10', 'candidate_recall']:
        print(f'{k:<20} {m[k]:.4f}')
    print(f"{'':<20} {'sebelum':>8} {'sesudah':>8}")
    for k in ['evidence_recall@5', 'hit@1', 'mrr']:
        after = m.get(f'after/{k}')
        after = '-' if after is None else f'{after:.4f}'
        print(f"{k:<20} {m[f'before/{k}']:>8.4f} {after:>8}")

In [ ]:
RUN_EVAL_TASK2 = True  # @param {type:"boolean"}
LIMIT = 0  # @param {type:"integer"}
# LIMIT 0 = semua 500 claim, isi misal 50 buat tes cepat

if RUN_EVAL_TASK2:
    claims = read_claims_cache(CLAIMS_SAMPLE_PATH)
    if LIMIT:
        claims = claims[:LIMIT]
    start = time.time()
    metrics, preds = evaluate_retrieval(claims, evidence_pipe.retriever, evidence_pipe.reranker)
    print_retrieval_metrics(metrics)
    print(f'{time.time() - start:.0f} detik')

    (OUTPUTS_DIR / 'task2').mkdir(exist_ok=True)
    (OUTPUTS_DIR / 'task2' / 'metrics.json').write_text(json.dumps({'n_claims': len(claims), **metrics}, indent=2))
    save_jsonl(preds, OUTPUTS_DIR / 'task2' / 'predictions.jsonl')

## Task 3 - Verification (NLI)
Model: `MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli` (zero-shot, ga ditrain lagi).
- premise = `"judul halaman: kalimat evidence"`, hypothesis = claim
- tiap kalimat evidence dinilai sendiri-sendiri, ditambah 1 pasangan lagi yang premise-nya gabungan semua kalimat (buat claim yang butuh 2 kalimat sekaligus)
- support = max P(entailment), refute = max P(contradiction). Kalau dua-duanya < threshold → NOT ENOUGH INFO, kalau ga ambil yang lebih gede

In [ ]:
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
LABELS = ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']
NLI_KEYS = ['entailment', 'neutral', 'contradiction']


def aggregate(sentence_probs, concat_probs=None, threshold=0.5):
    rows = [np.asarray(p, dtype=float) for p in sentence_probs]
    if concat_probs is not None:
        rows.append(np.asarray(concat_probs, dtype=float))
    if not rows:
        return 'NOT ENOUGH INFO', {'SUPPORTS': 0.0, 'REFUTES': 0.0, 'NOT ENOUGH INFO': 1.0}

    probs = np.stack(rows)
    support, refute = float(probs[:, 0].max()), float(probs[:, 2].max())
    scores = {'SUPPORTS': support, 'REFUTES': refute, 'NOT ENOUGH INFO': 1 - max(support, refute)}
    if max(support, refute) < threshold:
        return 'NOT ENOUGH INFO', scores
    return ('SUPPORTS' if support >= refute else 'REFUTES'), scores


class NLIVerifier:
    def __init__(self, model_name=NLI_MODEL, threshold=0.5, use_concat=True, batch_size=16):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.model_name = model_name
        self.threshold = threshold
        self.use_concat = use_concat
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()

        # urutan label tiap model NLI beda-beda, jadi ambil dari config
        id2label = {i: name.lower() for i, name in self.model.config.id2label.items()}
        self.cols = [next(i for i, name in id2label.items() if name.startswith(k[:7])) for k in NLI_KEYS]

    @torch.no_grad()
    def predict_pairs(self, premises, hypotheses):
        out = []
        for i in range(0, len(premises), self.batch_size):
            batch = self.tokenizer(premises[i:i + self.batch_size], hypotheses[i:i + self.batch_size],
                                   truncation='only_first', max_length=512, padding=True,
                                   return_tensors='pt').to(device)
            logits = self.model(**batch).logits.float()
            out.append(torch.softmax(logits, dim=-1)[:, self.cols].cpu().numpy())
        return np.concatenate(out) if out else np.zeros((0, 3))

    def score(self, claim, evidence):
        premises = [f"{ev['title']}: {clean_text(ev['text'])}" for ev in evidence]
        if not premises:
            return np.zeros((0, 3)), None
        if self.use_concat and len(premises) > 1:
            probs = self.predict_pairs(premises + [' '.join(premises)], [claim] * (len(premises) + 1))
            return probs[:-1], probs[-1]
        return self.predict_pairs(premises, [claim] * len(premises)), None

    def verify(self, claim, evidence):
        sentence_probs, concat_probs = self.score(claim, evidence)
        label, scores = aggregate(sentence_probs, concat_probs, self.threshold)
        scored = []
        for ev, p in zip(evidence, sentence_probs):
            ev = dict(ev)
            ev['nli'] = dict(zip(NLI_KEYS, map(float, p)))
            scored.append(ev)
        return {'claim': claim, 'label': label, 'confidence': scores[label], 'scores': scores, 'evidence': scored}

In [ ]:
verifier = NLIVerifier()

print('\n===== TASK 3: verification =====')
res = verifier.verify(demo['claim'], demo_evidence)
print('claim:', demo['claim'])
print(f"prediksi: {res['label']} ({res['confidence']:.3f}) | gold: {demo['label']}")
for ev in res['evidence']:
    nli = ev['nli']
    print(f"  E={nli['entailment']:.2f} N={nli['neutral']:.2f} C={nli['contradiction']:.2f}  [{ev['title']}] {ev['text']}")

### Evaluasi task 3
Data eval: 500 claim SUPPORTS/REFUTES dari `claims_sample.jsonl` + 250 claim NEI dari `shared_task_dev`.
Probabilitas NLI dihitung sekali, terus threshold NEI di-tuning di setengah data dan dilaporin di setengah sisanya.

Ada 2 mode:
- `retrieved`: evidence dari task 2 (pipeline beneran)
- `oracle`: pakai gold evidence buat claim S/R, jadi batas atas performa NLI-nya

FEVER score = label bener DAN (buat S/R) evidence set lengkap ada di top-k evidence.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

THRESHOLDS = [round(t, 2) for t in np.arange(0.30, 0.96, 0.05)]


def nli_metrics(rows, threshold):
    y_true, y_pred, fever = [], [], []
    for r in rows:
        label, _ = aggregate(r['sentence_probs'], r['concat_probs'], threshold)
        y_true.append(r['label'])
        y_pred.append(label)
        evidence_ok = r['label'] == 'NOT ENOUGH INFO' or full_set_found(r['gold_evidence'], r['evidence_keys'])
        fever.append(label == r['label'] and evidence_ok)

    p, rc, f, n = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    return {
        'threshold': threshold,
        'n': len(rows),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0),
        'fever_score': float(np.mean(fever)),
        'per_class': {lab: {'precision': p[i], 'recall': rc[i], 'f1': f[i], 'support': int(n[i])}
                      for i, lab in enumerate(LABELS)},
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=LABELS).tolist(),
    }


def print_nli_metrics(title, m):
    print(f"\n{title} (n={m['n']}, threshold={m['threshold']:.2f})")
    print(f"accuracy {m['accuracy']:.4f} | macro F1 {m['macro_f1']:.4f} | FEVER score {m['fever_score']:.4f}")
    for lab, s in m['per_class'].items():
        print(f"  {lab:<16} P={s['precision']:.3f} R={s['recall']:.3f} F1={s['f1']:.3f} n={s['support']}")
    print('  confusion matrix (baris = gold, kolom = prediksi S/R/NEI):', m['confusion_matrix'])


def evaluate_verification(evidence_pipe, verifier, oracle=False, n_nei=250, limit=None, seed=42, top_k=5):
    claims = read_claims_cache(CLAIMS_SAMPLE_PATH)
    nei = [c for c in load_claims(download_fever_file('shared_task_dev.jsonl'), include_nei=True)
           if c['label'] == 'NOT ENOUGH INFO']
    claims += random.Random(seed).sample(nei, min(n_nei, len(nei)))
    random.Random(seed).shuffle(claims)
    if limit:
        claims = claims[:limit]
    print(len(claims), 'claim:', {lab: sum(c['label'] == lab for c in claims) for lab in LABELS})

    # lookup (page, line) -> kalimat, buat mode oracle
    sentences = {}
    if oracle:
        for page in evidence_pipe.retriever.pages:
            for line_id, text in page['sentences']:
                sentences[(page['id'], int(line_id))] = {'title': page['title'], 'text': text}

    rows = []
    start = time.time()
    for c in tqdm(claims):
        evidence = evidence_pipe.run(c['claim'], top_k=top_k)
        if oracle and c['label'] != 'NOT ENOUGH INFO':
            # pakai gold evidence set pertama yang semua kalimatnya ada di corpus
            for ev_set in c['evidence_sets']:
                keys = [tuple(k) for k in ev_set]
                if all(k in sentences for k in keys):
                    evidence = [{'page_id': p, 'line_id': l, **sentences[(p, l)]} for p, l in keys]
                    break
        sentence_probs, concat_probs = verifier.score(c['claim'], evidence)
        rows.append({
            'id': c['id'], 'claim': c['claim'], 'label': c['label'], 'gold_evidence': c['evidence_sets'],
            'evidence_keys': [(e['page_id'], e['line_id']) for e in evidence],
            'evidence': [{k: e[k] for k in ['page_id', 'line_id', 'title', 'text']} for e in evidence],
            'sentence_probs': sentence_probs.tolist(),
            'concat_probs': None if concat_probs is None else concat_probs.tolist(),
        })
    elapsed = time.time() - start

    fixed = nli_metrics(rows, verifier.threshold)
    print_nli_metrics('semua claim, threshold default', fixed)

    # tuning threshold di setengah data, test di setengah sisanya
    half = len(rows) // 2
    sweep = [nli_metrics(rows[:half], t) for t in THRESHOLDS]
    best = max(sweep, key=lambda m: m['macro_f1'])
    print('\nthreshold sweep (macro F1 / FEVER):')
    for m in sweep:
        print(f"  {m['threshold']:.2f}  {m['macro_f1']:.4f}  {m['fever_score']:.4f}", '<- best' if m is best else '')
    tuned = nli_metrics(rows[half:], best['threshold'])
    print_nli_metrics('test half, threshold hasil tuning', tuned)
    print(f'{elapsed:.0f} detik')

    tag = 'oracle' if oracle else 'retrieved'
    out_dir = OUTPUTS_DIR / 'task3'
    out_dir.mkdir(exist_ok=True)
    summary = {'n_claims': len(rows), 'elapsed_s': elapsed, 'fixed_threshold': fixed,
               'tuned_threshold': best['threshold'], 'tuned_test_half': tuned}
    (out_dir / f'metrics_{tag}.json').write_text(json.dumps(summary, indent=2, default=float))
    for r in rows:
        r['predicted'], r['scores'] = aggregate(r['sentence_probs'], r['concat_probs'], verifier.threshold)
    save_jsonl(rows, out_dir / f'predictions_{tag}.jsonl')
    return fixed, tuned

In [ ]:
RUN_EVAL_RETRIEVED = True  # @param {type:"boolean"}
RUN_EVAL_ORACLE = True  # @param {type:"boolean"}
LIMIT = 0  # @param {type:"integer"}
# LIMIT 0 = semua 750 claim

if RUN_EVAL_RETRIEVED:
    print('===== eval task 3: retrieved evidence =====')
    evaluate_verification(evidence_pipe, verifier, oracle=False, limit=LIMIT)

if RUN_EVAL_ORACLE:
    print('\n===== eval task 3: oracle evidence =====')
    evaluate_verification(evidence_pipe, verifier, oracle=True, limit=LIMIT)

## Full Pipeline (Task 1 → 2 → 3)
Isi teks di `TEXT` (bahasa Inggris). Kalau kosong, dipakai contoh teks yang isinya salah satu claim dari sampel.
Inget corpus-nya cuma subset, jadi claim yang topiknya ga ada di corpus kemungkinan besar hasilnya NOT ENOUGH INFO.

In [ ]:
class FactCheckPipeline:
    def __init__(self, detector, evidence_pipe, verifier, top_k=5):
        self.detector = detector
        self.evidence_pipe = evidence_pipe
        self.verifier = verifier
        self.top_k = top_k

    def verify_claim(self, claim, p_claim=None):
        evidence = self.evidence_pipe.run(claim, top_k=self.top_k)
        result = self.verifier.verify(claim, evidence)
        return {'claim': claim, 'p_claim': p_claim, 'verdict': result['label'],
                'confidence': result['confidence'], 'scores': result['scores'], 'evidence': result['evidence']}

    def check(self, text):
        return [self.verify_claim(c['sentence'], c['p_claim']) for c in self.detector.detect(text)]


pipeline = FactCheckPipeline(detector, evidence_pipe, verifier)

In [ ]:
TEXT = ''  # @param {type:"string"}

text = TEXT.strip() or f"Good evening everyone, thanks for coming. {sample_claims[1]['claim']} I really love pizza here."
print('input:', text)

print('\n===== TASK 1: claim detection =====')
sentences = detector.detect(text, return_all=True)
for r in sentences:
    print('CLAIM' if r['is_claim'] else '  -  ', f"{r['p_claim']:.3f}", r['sentence'])

results = []
for c in [r for r in sentences if r['is_claim']]:
    r = pipeline.verify_claim(c['sentence'], c['p_claim'])
    results.append(r)

    print('\n===== TASK 2: evidence =====')
    print('claim:', r['claim'])
    for i, ev in enumerate(r['evidence'], 1):
        print(f"{i}. [{ev['title']} #{ev['line_id']}] {ev['rerank_score']:.2f}  {ev['text']}")

    print('\n===== TASK 3: verdict =====')
    print(f"{r['verdict']} ({r['confidence']:.3f})")

if not results:
    print('ga ada kalimat yang kedeteksi sebagai claim')

(OUTPUTS_DIR / 'pipeline').mkdir(exist_ok=True)
(OUTPUTS_DIR / 'pipeline' / 'last_run.json').write_text(json.dumps({'text': text, 'results': results}, indent=2))

## Hasil

In [ ]:
print('Task 1 (ClaimBuster test, threshold tuned)')
for path in MODELS_DIR.rglob('metrics.json'):
    m = json.loads(path.read_text())
    if 'test@tuned' in m:
        t = m['test@tuned']
        print(f"  {m['run']}: macroF1={t['macro_f1']:.4f} P={t['precision']:.4f} R={t['recall']:.4f} AUC={t['roc_auc']:.4f}")

print('\nTask 2')
path = OUTPUTS_DIR / 'task2' / 'metrics.json'
if path.exists():
    print_retrieval_metrics(json.loads(path.read_text()))
else:
    print('  belum dijalanin')

print('\nTask 3')
for tag in ['retrieved', 'oracle']:
    path = OUTPUTS_DIR / 'task3' / f'metrics_{tag}.json'
    if not path.exists():
        print(f'  {tag}: belum dijalanin')
        continue
    m = json.loads(path.read_text())
    for name, r in [('default', m['fixed_threshold']), (f"tuned {m['tuned_threshold']}", m['tuned_test_half'])]:
        print(f"  {tag:<9} {name:<12} acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} FEVER={r['fever_score']:.4f}")